## 1. Import Dependencies

In [1]:
# %pip install xgboost

In [58]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score

import xgboost as xgb

## 2. Load Cleaned Dataset

In [10]:
df = pd.read_csv('../data/processed/hotels_clean.csv')

In [11]:
df.shape

(84392, 27)

In [12]:
df.head().T

,0,1,2,3,4
hotel,resort_hotel,resort_hotel,resort_hotel,resort_hotel,resort_hotel
is_canceled,0,0,0,0,0
lead_time,7,13,14,0,9
arrival_date_year,2015,2015,2015,2015,2015
arrival_date_month,july,july,july,july,july
arrival_date_week_number,27,27,27,27,27
arrival_date_day_of_month,1,1,1,1,1
stays_in_weekend_nights,0,0,0,0,0
stays_in_week_nights,1,1,2,2,2
adults,1,1,2,2,2


In [13]:
df.dtypes

hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
deposit_type                          str
agent                               int64
days_in_waiting_list                int64
customer_type                     

In [16]:
df['is_canceled'].value_counts(normalize=True)

is_canceled
0    0.721016
1    0.278984
Name: proportion, dtype: float64

## 3. Split Training, Validation and Testing Data

In [17]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_canceled']) # stratify to preserve the cancellation proportions across the datasets.

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42, stratify=df_full_train['is_canceled'])

In [32]:
len(df_train) , len(df_val) , len(df_test) , len(df)

(50634, 16879, 16879, 84392)

In [35]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [26]:
y_train = df_train['is_canceled'].values
y_val = df_val['is_canceled'].values
y_test = df_test['is_canceled'].values

In [27]:
del df_train['is_canceled']
del df_val['is_canceled']
del df_test['is_canceled']

In [41]:
df_train

,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,days_in_waiting_list,customer_type,adr,total_of_special_requests,has_company
0,city_hotel,0,2016,january,2,3,1,0,1,0.0,...,0,0,a,no_deposit,9,0,transient,81.00,0,0
1,city_hotel,72,2016,april,17,23,2,2,2,0.0,...,0,0,a,no_deposit,9,0,transient,95.20,1,0
2,city_hotel,0,2017,june,26,26,1,3,2,0.0,...,0,0,a,no_deposit,14,0,transient,112.50,0,0
3,city_hotel,64,2017,july,30,29,0,1,2,0.0,...,0,0,a,no_deposit,9,0,transient,150.00,2,0
4,city_hotel,226,2017,march,10,7,0,2,2,0.0,...,0,0,a,no_deposit,83,0,transient,72.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50629,resort_hotel,115,2017,may,18,4,2,3,2,0.0,...,0,0,d,no_deposit,177,0,transient,56.00,0,0
50630,city_hotel,4,2016,august,36,29,1,0,3,0.0,...,0,0,d,no_deposit,9,0,transient,184.00,0,0
50631,resort_hotel,11,2015,august,31,1,4,10,1,0.0,...,0,0,a,no_deposit,104,0,transient,172.00,0,0
50632,city_hotel,298,2016,october,42,9,2,5,2,0.0,...,0,0,d,no_deposit,7,0,transient,79.59,1,0


## 4. Convert DataFrames into Dictionaries

In [57]:
train_dicts = df_train.to_dict(orient="records")

In [62]:
# Create DictVectorizer

dv = DictVectorizer(sparse=True)

In [63]:
# Learn feature names and transform training data

X_train = dv.fit_transform(train_dicts)

In [64]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1316484 stored elements and shape (50634, 224)>

In [46]:
dv.get_feature_names_out()

array(['adr', 'adults', 'agent', 'arrival_date_day_of_month',
       'arrival_date_month=april', 'arrival_date_month=august',
       'arrival_date_month=december', 'arrival_date_month=february',
       'arrival_date_month=january', 'arrival_date_month=july',
       'arrival_date_month=june', 'arrival_date_month=march',
       'arrival_date_month=may', 'arrival_date_month=november',
       'arrival_date_month=october', 'arrival_date_month=september',
       'arrival_date_week_number', 'arrival_date_year', 'babies',
       'children', 'country=abw', 'country=ago', 'country=alb',
       'country=and', 'country=are', 'country=arg', 'country=arm',
       'country=ata', 'country=atf', 'country=aus', 'country=aut',
       'country=aze', 'country=bel', 'country=ben', 'country=bgd',
       'country=bgr', 'country=bhr', 'country=bhs', 'country=bih',
       'country=blr', 'country=bol', 'country=bra', 'country=bwa',
       'country=caf', 'country=che', 'country=chl', 'country=chn',
       'countr

In [67]:
# Transform validation data using the same feature definitions

val_dicts = df_val.to_dict(orient="records")
X_val = dv.transform(val_dicts)

In [68]:
# Transform test data

test_dicts = df_test.to_dict(orient="records")
X_test = dv.transform(test_dicts)

## 5. Train a Basic XGBoost Model

In [70]:
# xgb.DMatrix() - wrap training data into a special data structure (XGBoost's data format).
# Allow s=xgboost to train faster.

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=features)

In [88]:
xgb_params = {
    "eta": 0.3, 
    "max_depth": 6,
    "min_child_weight": 1,
    
    "subsample": 1,
    "colsample_bytree": 1,

    "objective": "binary:logistic",
    "eval_metric": "auc",
    "nthread": 8,

    "seed": 42,
    "verbosity": 1     # 1 = print warning
}

model = xgb.train(xgb_params, dtrain, num_boost_round=10)   

In [89]:
y_pred = model.predict(dval)

In [90]:
roc_auc_score(y_val, y_pred)

0.8745329360219152

In [82]:
watchlist = [(dtrain, "train"), (dval, "val")]

In [ ]:
%%capture output

xgb_params = {
    "eta": 0.3, 
    "max_depth": 6,
    "min_child_weight": 1,
    
    "subsample": 1,
    "colsample_bytree": 1,

    "objective": "binary:logistic",
    "eval_metric": "auc",
    "nthread": 8,

    "seed": 42,
    "verbosity": 1    
}

model = xgb.train(xgb_params, 
                  dtrain, 
                  num_boost_round=200,
                  verbose_eval=5,
                  evals=watchlist,
                  early_stopping_rounds=10)     # early_stopping_rounds=10 means it tells XGBoost to stop if the validation AUC does not improve for 10 consecutive rounds. This help prevent unnecessary training & reduce overfitting.